# Translate `DeepPavlov/atis_intent_classification` to French and Spanish

Translates the English ATIS (Airline Travel Information System) intent-classification
dataset into French and Spanish using two locally-hosted OpenAI-compatible LLM servers
(e.g. vLLM/SGLang), and saves the result as one HF dataset with a **subset (config) per
language**. Each subset has 4 columns: `text` (original English), `label`, and one
translation column per model (`text_gemma`, `text_qwen`) so the two models' outputs can
be compared directly.

ATIS utterances are lowercase, largely unpunctuated speech-transcript-style queries
about flights ("i want to fly from boston at 838 am and arrive in denver at 1110 in
the morning") -- a different register from typed customer-support text, so the prompt
and few-shot examples below are tuned for that.

- `gemma` — `google/gemma-4-31B-it` on `http://localhost:8088/v1`
- `qwen`  — `Qwen/Qwen3.6-27B-FP8` on `http://localhost:8000/v1`

**Setup.** This repo's `uv` environment already has `datasets`; it does not have
`openai`. Launch this notebook with the extra dependency pulled in on the fly, without
touching `pyproject.toml`:

```bash
uv run --with openai --with ipykernel jupyter lab
```

Translation is checkpointed to `translations/atis/*.jsonl` (one line per example),
so the notebook is safe to interrupt and re-run — already-translated rows are skipped.


In [ ]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from tqdm.auto import tqdm


## Config

Both models translate into both languages. Adjust ports/model names/languages here.

In [ ]:
MODELS = {
    "gemma": {"base_url": "http://localhost:8088/v1", "model": "google/gemma-4-31B-it"},
    "qwen": {
        "base_url": "http://localhost:8000/v1",
        "model": "Qwen/Qwen3.6-27B-FP8",
        # Qwen3 is a hybrid-thinking model: without this it emits its chain-of-thought
        # as the actual response content instead of a final answer.
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
}

LANGUAGES = {
    "fr": "French",
    "es": "Spanish",
}

OUT_DIR = Path("translations/atis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 64
TEMPERATURE = 0.0


In [ ]:
clients = {name: OpenAI(base_url=cfg["base_url"], api_key="EMPTY") for name, cfg in MODELS.items()}

for name, client in clients.items():
    available = [m.id for m in client.models.list().data]
    print(f"{name} ({MODELS[name]['base_url']}): serving {available}")
    assert MODELS[name]["model"] in available, (
        f"{MODELS[name]['model']} not found on {name} server; available: {available}"
    )


## Load the dataset

In [ ]:
raw = load_dataset("DeepPavlov/atis_intent_classification")
raw


## Translation

The prompt asks for a natural, native-sounding translation that preserves the register
of the source: ATIS utterances are lowercase, largely unpunctuated, speech-like queries
about flights, fares, airlines, and ground transportation. Translations should read like
the same kind of spoken-style query in the target language -- not a cleaned-up, capitalized,
fully-punctuated sentence. A few hand-written examples are included as few-shot
demonstrations, per target language, to anchor that register plus airline terminology
(city names, times, flight-booking phrasing).


In [ ]:
# Few-shot examples per language: (english, translation). Kept lowercase and lightly
# punctuated to match the register of ATIS utterances (speech-transcript-style queries).
# Time digits are kept exactly as in the source (e.g. "838", "1110") -- NOT reformatted
# into locale time conventions (not "8h38"/"14 heures"/"8:38"). The "am"/"pm" marker is
# dropped rather than translated literally, since plain am/pm is an English borrowing and
# not standard in French or Spanish.
EXAMPLES = {
    "fr": [
        (
            "i want to fly from boston at 838 am and arrive in denver at 1110 in the morning",
            "je veux prendre un vol au depart de boston a 838 et arriver a denver a 1110 le matin",
        ),
        (
            "what flights are available from pittsburgh to baltimore on thursday morning",
            "quels vols sont disponibles de pittsburgh a baltimore jeudi matin",
        ),
        (
            "show me all flights from boston to pittsburgh on wednesday of next week which leave boston after 2 o'clock pm",
            "montre-moi tous les vols de boston a pittsburgh mercredi de la semaine prochaine qui partent de boston apres 2",
        ),
    ],
    "es": [
        (
            "i want to fly from boston at 838 am and arrive in denver at 1110 in the morning",
            "quiero volar desde boston a las 838 y llegar a denver a las 1110 de la manana",
        ),
        (
            "what flights are available from pittsburgh to baltimore on thursday morning",
            "que vuelos hay disponibles de pittsburgh a baltimore el jueves por la manana",
        ),
        (
            "show me all flights from boston to pittsburgh on wednesday of next week which leave boston after 2 o'clock pm",
            "muestrame todos los vuelos de boston a pittsburgh el miercoles de la semana que viene que salen de boston despues de las 2",
        ),
    ],
}


In [ ]:
SYSTEM_PROMPT = (
    "You are a professional translator localizing airline-travel-booking queries (ATIS "
    "system, speech-transcript-style: lowercase, little to no punctuation). Translate the "
    "user's message from English into {lang_name}. Keep the same meaning, tone, and "
    "register -- including the lowercase, loosely-punctuated, spoken-query style -- but "
    "produce something that reads naturally to a native {lang_name} speaker, using correct "
    "airline/travel terminology (city names, times, flights, fares, airlines). "
    "Keep time digits in their raw source format exactly as written (e.g. '838', '1110') -- "
    "do NOT reformat them into locale time conventions (e.g. do not turn '838' into '8h38' "
    "or '8:38'). Drop the 'am'/'pm' marker rather than translating it literally, since plain "
    "am/pm is an English borrowing, not standard in {lang_name} (e.g. '838 am' -> '838'; "
    "'2 pm' -> '2'). "
    "Do not add, remove, or explain anything. "
    "Reply with ONLY the translated sentence: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_examples_block(lang_code):
    lines = []
    for en, translated in EXAMPLES.get(lang_code, []):
        lines.append(f"EN: {en}\n{lang_code.upper()}: {translated}")
    return "\n\n".join(lines)


THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
# Heuristics for a hybrid-thinking model leaking its chain-of-thought as plain text
# (no <think> tags) instead of -- or in addition to -- a final answer.
REASONING_MARKERS = re.compile(
    r"^\s*(here'?s a thinking process|let'?s (think|analyze)|step \d|\d+\.\s+\*\*)",
    re.IGNORECASE,
)


def clean_translation(raw_out):
    out = THINK_RE.sub("", raw_out).strip()
    out = out.strip('"').strip("'").strip()
    return out


def translate_one(client, model, text, lang_code, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = SYSTEM_PROMPT.format(lang_name=lang_name, examples_block=build_examples_block(lang_code))
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": text},
    ]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                extra_body=extra_body or {},
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning, not a translation: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Translation failed for {text!r}: {last_err}")


### Checkpointed, concurrent translation of a whole split, by one model, into one language

In [ ]:
def translate_split(job_name, dataset, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    out_path = OUT_DIR / f"{job_name}_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[row["idx"]] = row

    todo = [i for i in range(len(dataset)) if i not in done]
    print(f"[{job_name}/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_one, client, model, dataset[i]["text"], lang_code, extra_body): i
                for i in todo
            }
            desc = f"{model_key}"
            bar = tqdm(as_completed(futures), total=len(futures), desc=desc, position=position, leave=True)
            for fut in bar:
                bar.set_description(f"{model_key}: {job_name}/{lang_code}")
                i = futures[fut]
                text_translated = fut.result()
                row = {
                    "idx": i,
                    "text": dataset[i]["text"],
                    "text_translated": text_translated,
                    "label": dataset[i]["label"],
                    "label_text": dataset[i]["label_text"],
                }
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[i] = row

    return [done[i] for i in range(len(dataset))]


## Smoke test

Translate a handful of examples with both models before committing to a full run.

In [ ]:
sample = raw["test"].select(range(5))
for lang_code in LANGUAGES:
    for model_key in MODELS:
        rows = translate_split("smoketest", sample, lang_code, model_key)
        for r in rows:
            print(f"[{lang_code}/{model_key}] {r['text']!r}  ->  {r['text_translated']!r}")
        print()


## Full run

Each model hits its own server, so gemma and qwen run in parallel (one thread per
model), each working through `test` (faster feedback) then `train`, both languages.
Within a model, requests to its server are still capped at `MAX_WORKERS` concurrent.
Each model gets a fixed progress-bar row (`position`) so the two bars update in place
side by side instead of clobbering each other's output -- they will very likely finish
at different times since the two servers/models have different throughput.
Safe to re-run / resume — already-translated rows are skipped.


In [ ]:
def run_model_jobs(model_key, position):
    results = {}
    for split_name in ["test", "train"]:
        for lang_code in LANGUAGES:
            results[(split_name, lang_code, model_key)] = translate_split(
                split_name, raw[split_name], lang_code, model_key, position=position
            )
    return results


translated = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {
        ex.submit(run_model_jobs, model_key, position): model_key
        for position, model_key in enumerate(MODELS)
    }
    for fut in as_completed(futures):
        translated.update(fut.result())


## Assemble into one dataset per language (4 columns: `text`, `label`, `text_gemma`, `text_qwen`)


In [ ]:
model_keys = list(MODELS)  # e.g. ["gemma", "qwen"]

final = {}
for lang_code in LANGUAGES:
    dd = DatasetDict()
    for split_name in ["train", "test"]:
        by_model = {mk: translated[(split_name, lang_code, mk)] for mk in model_keys}
        n = len(by_model[model_keys[0]])
        rows = []
        for i in range(n):
            row = {
                "text": by_model[model_keys[0]][i]["text"],
                "label": by_model[model_keys[0]][i]["label"],
            }
            for mk in model_keys:
                row[f"text_{mk}"] = by_model[mk][i]["text_translated"]
            rows.append(row)
        dd[split_name] = Dataset.from_list(rows)
    final[lang_code] = dd
    print(lang_code, dd)


## Spot-check quality

In [ ]:
lang_code = "fr"
split_name = "test"
idxs = random.sample(range(len(final[lang_code][split_name])), 10)
for i in idxs:
    row = final[lang_code][split_name][i]
    print("EN:", row["text"])
    for mk in MODELS:
        print(f"{lang_code.upper()} [{mk}]:", row[f"text_{mk}"])
    print("label:", row["label"])
    print()


## Save as one HF dataset with a subset per language

Locally, each language becomes its own `save_to_disk` directory (mirrors a subset).
To publish as a single dataset repo with per-language **configs**, push each
`DatasetDict` with `config_name=lang_code` -- left commented out, uncomment and set
your own repo id if you want to publish.


In [ ]:
SAVE_DIR = Path("translations/atis_final")
for lang_code, dd in final.items():
    dd.save_to_disk(str(SAVE_DIR / lang_code))

# repo_id = "<your-username>/atis-mt"
# for lang_code, dd in final.items():
#     dd.push_to_hub(repo_id, config_name=lang_code)
